# Session 11: Working with APIs

**Course:** Python for Data Engineering  
**Phase 3:** Data Engineering Concepts

**What we'll cover:**
- REST API basics (GET, POST, status codes)
- Making requests with the `requests` library
- Handling JSON responses
- Pagination and rate limiting
- Storing API data into structured format

**Note:** This session is demo-heavy. Follow along by running each cell. We'll use free public APIs.

In [ ]:
import requests
import json
import pandas as pd
import time

---

## 1. REST API Basics

APIs let you fetch data from external services over HTTP. As a data engineer, you'll pull data from APIs into your pipelines.

| HTTP Method | Purpose | Example |
|-------------|---------|--------|
| `GET` | Retrieve data | Get a list of users |
| `POST` | Send data | Submit a new record |
| `PUT` | Update data | Update a user's email |
| `DELETE` | Remove data | Delete a record |

For data engineering, you'll mostly use **GET** to extract data.

### HTTP Status Codes

| Code | Meaning | What to do |
|------|---------|------------|
| 200 | OK | Data returned successfully |
| 201 | Created | Record was created |
| 400 | Bad Request | Fix your request parameters |
| 401 | Unauthorized | Check your API key |
| 404 | Not Found | Wrong endpoint or ID |
| 429 | Too Many Requests | Slow down — rate limited |
| 500 | Server Error | API is broken — retry later |

---

## 2. Making Requests

We'll use [JSONPlaceholder](https://jsonplaceholder.typicode.com) — a free fake API for testing.

In [ ]:
# Basic GET request

response = requests.get("https://jsonplaceholder.typicode.com/users")

print(f"Status code: {response.status_code}")
print(f"Content type: {response.headers['Content-Type']}")
print(f"Response size: {len(response.text)} chars")

In [ ]:
# Parse JSON response

users = response.json()  # converts JSON string to Python list/dict
print(f"Got {len(users)} users")
print(f"Type: {type(users)}")

# Look at the first user
print(json.dumps(users[0], indent=2))

In [ ]:
# Convert to DataFrame — the usual end goal

df_users = pd.DataFrame(users)
print(df_users[["id", "name", "email", "phone"]].to_string())

In [ ]:
# Fetch a specific resource by ID

user_id = 1
response = requests.get(f"https://jsonplaceholder.typicode.com/users/{user_id}")
user = response.json()
print(f"User: {user['name']}")
print(f"Email: {user['email']}")
print(f"City: {user['address']['city']}")
print(f"Company: {user['company']['name']}")

In [ ]:
# Query parameters — filtering on the server side

# Get posts by a specific user
params = {"userId": 1}
response = requests.get("https://jsonplaceholder.typicode.com/posts", params=params)
posts = response.json()

print(f"Posts by user 1: {len(posts)}")
for post in posts[:3]:
    print(f"  [{post['id']}] {post['title'][:50]}...")

---

## 3. Handling Errors and Retries

APIs fail. Networks drop. Rate limits hit. Always handle errors.

In [ ]:
# Always check status code before using the response

def fetch_data(url, params=None):
    """Fetch data from API with error handling."""
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()  # raises exception for 4xx/5xx
        return response.json()
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error: {e}")
        return None
    except requests.exceptions.ConnectionError:
        print("Connection failed")
        return None
    except requests.exceptions.Timeout:
        print("Request timed out")
        return None

# Test with valid URL
data = fetch_data("https://jsonplaceholder.typicode.com/posts/1")
print(f"Success: {data['title'][:40]}...")

# Test with invalid URL
data = fetch_data("https://jsonplaceholder.typicode.com/invalid")
print(f"Failed: {data}")

In [ ]:
# Retry pattern — attempt multiple times with backoff

def fetch_with_retry(url, max_retries=3, backoff=1):
    """Fetch with exponential backoff retry."""
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            print(f"  Attempt {attempt}: success")
            return response.json()
        except Exception as e:
            print(f"  Attempt {attempt}: failed ({e})")
            if attempt < max_retries:
                wait = backoff * (2 ** (attempt - 1))  # 1s, 2s, 4s...
                print(f"  Waiting {wait}s before retry...")
                time.sleep(wait)
    
    print("  All retries exhausted")
    return None

# This should succeed on first try
data = fetch_with_retry("https://jsonplaceholder.typicode.com/posts/1")

---

## 4. Handling Nested JSON

API responses are often deeply nested. You need to flatten them for tabular storage.

In [ ]:
# Users have nested address and company objects
users = requests.get("https://jsonplaceholder.typicode.com/users").json()

# Method 1: json_normalize flattens nested dicts
df_flat = pd.json_normalize(users)
print(f"Columns: {list(df_flat.columns)}")
df_flat[["id", "name", "email", "address.city", "company.name"]].head()

In [ ]:
# Method 2: Manual flattening — more control

flat_users = []
for user in users:
    flat_users.append({
        "id": user["id"],
        "name": user["name"],
        "email": user["email"],
        "phone": user["phone"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "company": user["company"]["name"],
    })

df_manual = pd.DataFrame(flat_users)
df_manual.head()

---

## 5. Pagination

Most APIs don't return all data at once. They use pagination — you fetch page by page.

In [ ]:
# Simulating pagination with JSONPlaceholder
# The API doesn't paginate, but we'll use _start and _limit params

def fetch_all_pages(base_url, page_size=10):
    """Fetch all pages from a paginated API."""
    all_data = []
    page = 0
    
    while True:
        params = {"_start": page * page_size, "_limit": page_size}
        response = requests.get(base_url, params=params)
        data = response.json()
        
        if not data:  # empty response = no more pages
            break
        
        all_data.extend(data)
        print(f"  Page {page + 1}: got {len(data)} records (total: {len(all_data)})")
        page += 1
        
        time.sleep(0.2)  # be nice to the API
    
    return all_data

all_posts = fetch_all_pages("https://jsonplaceholder.typicode.com/posts", page_size=20)
print(f"\nTotal posts fetched: {len(all_posts)}")

---

## 6. Full API-to-File Pipeline

Putting it all together — fetch from API, flatten, clean, save.

In [ ]:
import logging

log = logging.getLogger("api_pipeline")
log.setLevel(logging.INFO)
log.handlers.clear()
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s", datefmt="%H:%M:%S"))
log.addHandler(handler)

def api_to_csv_pipeline():
    """Extract users from API, flatten, and save to CSV."""
    log.info("Pipeline started")
    
    # Extract
    log.info("Fetching users from API")
    response = requests.get("https://jsonplaceholder.typicode.com/users", timeout=10)
    response.raise_for_status()
    users = response.json()
    log.info(f"Fetched {len(users)} users")
    
    # Transform — flatten nested JSON
    log.info("Flattening and cleaning data")
    flat = []
    for u in users:
        flat.append({
            "id": u["id"],
            "name": u["name"],
            "username": u["username"],
            "email": u["email"].lower(),
            "phone": u["phone"].split(" ")[0],  # clean phone
            "city": u["address"]["city"],
            "company": u["company"]["name"],
        })
    
    df = pd.DataFrame(flat)
    
    # Load
    output_path = "data/api_users.csv"
    df.to_csv(output_path, index=False)
    log.info(f"Saved {len(df)} records to {output_path}")
    log.info("Pipeline complete")
    
    return df

result = api_to_csv_pipeline()
result

---

## Lab: Build an API Data Pipeline

Build a pipeline that:

1. Fetch all **posts** from `https://jsonplaceholder.typicode.com/posts`
2. Fetch all **users** from `https://jsonplaceholder.typicode.com/users`
3. Join posts with users to add `author_name` and `author_email` to each post
4. Calculate: number of posts per user
5. Save the enriched posts to `data/api_posts_enriched.csv`
6. Save the per-user summary to `data/api_user_post_summary.json`
7. Add logging and error handling

In [ ]:
import requests
import pandas as pd
import json
import logging

# Your code here


---

## Summary

| Topic | Key Takeaway |
|-------|--------------|
| `requests.get()` | Fetch data from APIs, returns a Response object |
| `.json()` | Parse response body as Python dict/list |
| Status codes | Always check — 200 = ok, 4xx = your fault, 5xx = server fault |
| Error handling | `raise_for_status()`, try/except, retry with backoff |
| Nested JSON | `pd.json_normalize()` or manual flattening |
| Pagination | Loop until empty response, use `_start`/`_limit` or `page` params |
| Rate limiting | `time.sleep()` between requests — be nice to APIs |

**Key patterns:**
- Always set a `timeout` on requests
- Flatten nested JSON before storing as tabular data
- Retry with exponential backoff for transient failures
- Sleep between requests to avoid rate limits

**Next session:** Database Integration — connecting Python to databases, inserting and querying data.